# Exploration: Aeolus query endpoints

Notebook d'exploration pour:
- utiliser un token Aeolus existant **ou** générer un token OAuth via `client_id/client_secret`,
- appeler les endpoints de lecture avec `AeolusClient`:
  - `GET /assets`
  - `GET /portfolios`
  - `GET /countries`
  - `GET /bps`
- vérifier rapidement le statut, la structure et la taille des réponses.


In [5]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from src.aeolus_client import AeolusClient
from src.integrations.aeolus_publish_service import resolve_auth_config


In [6]:
# Charger .env.local si nécessaire (optionnel)
# Remarque: Jupyter peut ne pas démarrer à la racine repo.
repo_root = Path.cwd()
env_local = repo_root / '.env.local'
if env_local.exists():
    for raw_line in env_local.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

AEOLUS_BASE_URL = os.getenv('AEOLUS_BASE_URL', 'https://e6.aeolus.main.e6-group.com/api/v2')
AEOLUS_TOKEN_URL = os.getenv('AEOLUS_TOKEN_URL', 'https://eole-api-gateway-prod.auth.eu-west-1.amazoncognito.com/oauth2/token')

# Option 1: token direct
AEOLUS_ACCESS_TOKEN = os.getenv('AEOLUS_ACCESS_TOKEN')

# Option 2: OAuth client credentials
AEOLUS_CLIENT_ID = os.getenv('AEOLUS_CLIENT_ID')
AEOLUS_CLIENT_SECRET = os.getenv('AEOLUS_CLIENT_SECRET')

# Scopes lecture utiles pour cette exploration
AEOLUS_ALLOWED_SCOPES = [
    'https://aeolus.main.e6-group.com/read:assets',
    'https://aeolus.main.e6-group.com/read:portfolios',
    'https://aeolus.main.e6-group.com/read:countries',
    'https://aeolus.main.e6-group.com/read:bps',
]

AEOLUS_TOKEN_SCOPE = os.getenv('AEOLUS_TOKEN_SCOPE', ' '.join(AEOLUS_ALLOWED_SCOPES))

print('AEOLUS_BASE_URL:', AEOLUS_BASE_URL)
print('AEOLUS_TOKEN_URL:', AEOLUS_TOKEN_URL)
print('Using direct token:', bool(AEOLUS_ACCESS_TOKEN))
print('Using client credentials:', bool(AEOLUS_CLIENT_ID and AEOLUS_CLIENT_SECRET))


AEOLUS_BASE_URL: https://e6.aeolus.main.e6-group.com/api/v2
AEOLUS_TOKEN_URL: https://eole-api-gateway-prod.auth.eu-west-1.amazoncognito.com/oauth2/token
Using direct token: False
Using client credentials: True


In [7]:
# Reuse project auth resolver (same logic as API layer)
# - reads AEOLUS_* env vars
# - supports AEOLUS_ACCESS_TOKEN or AEOLUS_CLIENT_ID/AEOLUS_CLIENT_SECRET
auth_config = resolve_auth_config(None)
print('Auth config resolved via aeolus_publish_service.resolve_auth_config')


In [8]:
async def fetch_endpoints_independently() -> dict[str, dict[str, object]]:
    results: dict[str, dict[str, object]] = {}
    async with AeolusClient(base_url=AEOLUS_BASE_URL, auth=auth_config) as client:
        calls = [
            ('assets', client.get_assets),
            ('portfolios', client.get_portfolios),
            ('countries', client.get_countries),
            ('bps', client.get_bps),
        ]

        for name, endpoint_call in calls:
            try:
                payload = await endpoint_call()
                results[name] = {'ok': True, 'payload': payload, 'error': None}
                print(f'[OK] {name}')
            except Exception as exc:
                results[name] = {'ok': False, 'payload': None, 'error': str(exc)}
                print(f'[ERROR] {name}: {exc}')

    return results


results = await fetch_endpoints_independently()

assets_payload = results['assets']['payload']
portfolios_payload = results['portfolios']['payload']
countries_payload = results['countries']['payload']
bps_payload = results['bps']['payload']

print('Calls completed. Check `results` for per-endpoint status and errors.')


[ERROR] assets: Aeolus API error 400
[ERROR] portfolios: Aeolus API error 400
[ERROR] countries: Aeolus API error 400
[ERROR] bps: Aeolus API error 400
Calls completed. Check `results` for per-endpoint status and errors.


In [9]:
print('GET /assets via AeolusClient')
print('Success:', results['assets']['ok'])
if not results['assets']['ok']:
    print('Error:', results['assets']['error'])
else:
    print('Top-level type:', type(assets_payload).__name__)
    if isinstance(assets_payload, dict):
        print('Top-level keys:', list(assets_payload.keys()))
    print('Payload size (bytes):', sys.getsizeof(assets_payload))


GET /assets via AeolusClient
Success: False
Error: Aeolus API error 400


In [10]:
print('GET /portfolios via AeolusClient')
print('Success:', results['portfolios']['ok'])
if not results['portfolios']['ok']:
    print('Error:', results['portfolios']['error'])
else:
    print('Top-level type:', type(portfolios_payload).__name__)
    if hasattr(portfolios_payload, 'model_dump'):
        portfolios_dict = portfolios_payload.model_dump()
        print('Top-level keys:', list(portfolios_dict.keys()))
    print('Payload size (bytes):', sys.getsizeof(portfolios_payload))


GET /portfolios via AeolusClient
Success: False
Error: Aeolus API error 400


In [11]:
print('GET /countries via AeolusClient')
print('Success:', results['countries']['ok'])
if not results['countries']['ok']:
    print('Error:', results['countries']['error'])
else:
    print('Top-level type:', type(countries_payload).__name__)
    if isinstance(countries_payload, list):
        print('Items count:', len(countries_payload))
    print('Payload size (bytes):', sys.getsizeof(countries_payload))


GET /countries via AeolusClient
Success: False
Error: Aeolus API error 400


In [12]:
print('GET /bps via AeolusClient')
print('Success:', results['bps']['ok'])
if not results['bps']['ok']:
    print('Error:', results['bps']['error'])
else:
    print('Top-level type:', type(bps_payload).__name__)
    if isinstance(bps_payload, list):
        print('Items count:', len(bps_payload))
    print('Payload size (bytes):', sys.getsizeof(bps_payload))


GET /bps via AeolusClient
Success: False
Error: Aeolus API error 400


In [13]:
# Apercu brut (decommenter au besoin)
# assets_payload
# portfolios_payload
# countries_payload
# bps_payload
